# 40 — YOLO11n@640 data ablation (50 epochs)
**This is a data ablation, NOT the final V6.5-vs-V6.4 benchmark.**
V6.2E TRAIN + reviewed MPCD/PowerEquipment TRAIN + approved synthetic, evaluated on the
**frozen V6.2E VAL**. TEST is never materialized.
`yolo11n.pt`, imgsz 640, epochs 50, batch -1, device 0, seed 20261007.

Resume: if `runs/yolo11n_640/<run>/weights/last.pt` exists with a matching train hash,
training resumes from it (also restorable from Camber). Needs a CUDA GPU.

In [ ]:
# Bootstrap: make the project package importable from notebooks/.
import os, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "configs").is_dir())
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("YOLOV11SDI_PROJECT_ROOT", str(ROOT))

import pandas as pd
from IPython.display import display, Image as IPImage
from yolov11sdi import pipeline
from yolov11sdi.config import load_config
from yolov11sdi.paths import ProjectPaths
from yolov11sdi.state import StateStore

FORCE_REBUILD = False   # True = rebuild even if a matching complete result exists
DRY_RUN = False
paths = ProjectPaths.detect().ensure()
store = StateStore(paths.state)
print("env:", paths.env.value, "| home:", paths.home, "| data:", paths.data_root)

In [ ]:
pipeline.run_stage("yolo11n_ablation", dry_run=True)

In [ ]:
outcome = pipeline.run_stage("yolo11n_ablation", force=FORCE_REBUILD, dry_run=DRY_RUN)
print(outcome)
display(pd.Series(outcome.summary, dtype=object).to_frame("value"))

In [ ]:
from yolov11sdi.reporting.status import latest_checkpoint
print(latest_checkpoint(paths))
runs = sorted(paths.runs.glob("yolo11n_640/*/results.csv"))
if runs:
    r = pd.read_csv(runs[-1]); r.columns = [c.strip() for c in r.columns]
    display(r.tail())
    r.plot(x="epoch", y=[c for c in r.columns if "mAP50" in c], figsize=(8, 4))